In [1]:
%pip install --upgrade --quiet pdfplumber
%pip install --upgrade --quiet tiktoken
%pip install --upgrade --quiet openai
%pip install --upgrade --quiet python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import time
import base64
import tiktoken
import pdfplumber
import pandas as pd
from openai import AzureOpenAI
from dotenv import load_dotenv

In [ ]:
# Nome das regiões Azure OpenAI
EASTUS = "eastus"
CANADAEAST = "canadaeast"
NORTHCENTRALUS = "northcentralus"

# Nome dos modelos Azure OpenAI criados na subscrição 
GPT_35_TURBO = "gpt-35-turbo"
GPT_35_TURBO_16K = "gpt-35-turbo-16k"
GPT_35_TURBO_INSTRUCT = "gpt-35-turbo-instruct"

GPT_4 = "gpt-4"
GPT_4_32K = "gpt-4-32k"
GPT_4_TURBO = "gpt-4-turbo"
GPT_4o = "gpt-4o"
GPT_4o_mini = "gpt-4o-mini"

DALL_E_3 = "dall-e-3"
WHISPER = "whisper"
TEXT_EMBEDDING_ADA_002 = "text-embedding-ada-002"
TEXT_EMBEDDING_ADA_3_SMALL = "text-embedding-3-small"
TEXT_EMBEDDING_ADA_3_LARGE = "text-embedding-3-large"

# Whisper limit 25MB files
audio_chunk_size_kb = 1024 * 22
working_directory = os.path.dirname(os.path.abspath("."))

load_dotenv() # carregar variáveis de ambiente


def extract_table_as_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = ""
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

deployments_in_regions = {"CANADAEAST": [GPT_35_TURBO, GPT_4, GPT_4_32K, GPT_4_TURBO, TEXT_EMBEDDING_ADA_002, TEXT_EMBEDDING_ADA_3_SMALL],
                          "EASTUS": [GPT_35_TURBO_16K, GPT_35_TURBO_INSTRUCT, GPT_4o, GPT_4o_mini, DALL_E_3, TEXT_EMBEDDING_ADA_002, TEXT_EMBEDDING_ADA_3_LARGE],
                          "NORTHCENTRALUS": [WHISPER, GPT_4o]}


# carregar tokenizador para os modelos de linguagem
encoding = tiktoken.get_encoding("cl100k_base")

# inicialização do cliente Azure OpenAI Canada East
client_canadaeast = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_CANADAEAST"), 
    api_key=os.getenv("AZURE_OPENAI_KEY_CANADAEAST"),  
    api_version="2024-02-15-preview"
)

# inicialização do cliente Azure OpenAI East US
client_eastus = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_EASTUS"), 
    api_key=os.getenv("AZURE_OPENAI_KEY_EASTUS"),  
    api_version="2024-02-15-preview"
)

# inicialização do cliente Azure OpenAI North Central US
client_northcentralus = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_NORTHCENTRALUS"), 
    api_key=os.getenv("AZURE_OPENAI_KEY_NORTHCENTRALUS"),  
    api_version=os.getenv("WHISPER_VERSION")
)


# funcao para retornar o cliente de acordo com a região
def get_client(region):
    if region == "canadaeast":
        return client_canadaeast
    elif region == "eastus":
        return client_eastus
    elif region == "northcentralus":
        return client_northcentralus
    else:
        return None
def call_llm(deployment_name, system_message, prompt, file_path, region="canadaeast"):

    if deployment_name not in deployments_in_regions[region.upper()]:
        return {"0.error": f"Model {deployment_name} not available in region {region}", 
                "1.This regions has the following models available": deployments_in_regions[region.upper()], 
                f"2.The model {deployment_name} is available in the following regions": [region for region, models in deployments_in_regions.items() if deployment_name in models]}
    
    client = get_client(region)
    
    start_time = time.perf_counter()
    
    messages = [{"role": "system", "content": system_message},{"role": "user", "content": prompt}]
    
    if file_path:
        print("file_path", file_path)
        messages = [{"role": "system", "content": system_message},
                    {"role": "user", "content": [{"type": "text", "text": prompt}, get_content_file(file_path)]}]
    
    response = client.chat.completions.create(
        model = deployment_name,
        messages = messages
    )
    elapsed_time = time.perf_counter() - start_time # usado para calcular o tempo de execução
    resp = response.choices[0].message.content
    tokens_completion = num_tokens_from_string(resp, deployment_name) # conta os tokens da resposta
    # conta os tokens da mensagem do sistema e da pergunta (prompt)
    tokens_prompt = num_tokens_from_string(system_message, deployment_name) + num_tokens_from_string(prompt, deployment_name)
    # formatar a resposta da função
    return {"0.model": deployment_name, 
            "1.elapsed_time": elapsed_time, 
            "2.response": resp, 
            "3.num_tokens_completion": tokens_completion, 
            "4.num_tokens_prompt": tokens_prompt,
            "5.system_message": system_message,
            "6.prompt": prompt}

def get_content_file(file_path):
    response = {}
    if file_path.endswith(".wav"):
        with open(file_path, "rb") as audio_file:
            response = {
              "type": "audio_url",
              "audio_url": {
                "url": f"data:audio/wav;base64,{base64.b64encode(audio_file.read()).decode()}",
                "detail": "auto"
              }
            }
    elif file_path.endswith(".jpg") or file_path.endswith(".jpeg") or file_path.endswith(".png"):
        with open(file_path, "rb") as image_file:
            response = {
                    "type": "image_url",
                    "image_url": {
                        "url":  f"data:image/jpeg;base64,{base64.b64encode(image_file.read()).decode('utf-8')}",
                        "detail": "auto"
                    }
                }
    elif file_path.endswith(".pdf"):
        with open(file_path, "rb") as pdf_file:
            response = {
                "type": "text",
                "file": {
                    "url": f"data:text/pdf;base64,{base64.b64encode(pdf_file.read()).decode()}",
                    "detail": "auto"
                }
            }
    else:
        response = {
            "type": "file",
            "file": {
                "url": f"data:file/octet-stream;base64,{base64.b64encode(open(file_path, 'rb').read()).decode()}",
                "detail": "auto"
            }
        }
    return response

# funcao para contar tokens
def num_tokens_from_string(texto, model):
    if not texto:
        return 0
    encoding = tiktoken.encoding_for_model(model)
    num_tokens = len(encoding.encode(texto))
    return num_tokens


def hold_llm_response(llm_response):
    # remove o prefixo "```python" se existir
    llm_response = re.sub(r"^```html\s*", "", llm_response, flags=re.MULTILINE)
    # remove o sufixo "```" se existir
    llm_response = re.sub(r"```$", "", llm_response, flags=re.MULTILINE)
    if isinstance(llm_response, str):
        try:
            llm_response = llm_response.strip()
            return llm_response
        except Exception as e:
            print("Erro ao decodificar texto :", e)
            llm_response = []
    else:
        print("Erro: a resposta não é uma string válida.")

# funcao que recebe um conteudo CSV e grava em um arquivo CSV
def save_file(content, file_path):
    try:
        print("save_file", file_path)
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(hold_llm_response(content))
        return True
    except Exception as e:
        print(f"Erro ao salvar o arquivo: {e}")
        return False

# funcao que varre o diretório e retorna os arquivos PDF
def get_files_from_directory(directory_path, file_extension):
    files = []
    for root, dirs, filenames in os.walk(directory_path):
        for filename in filenames:
            if filename.endswith(file_extension):
                files.append(os.path.join(root, filename))
    return files

#funcao para ler um prompt de um arquivo pelo nome do arquivo
def read_prompt_from_file(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            prompt = file.read()
        return prompt
    except Exception as e:
        print(f"Erro ao ler o arquivo: {e}")
        return None
    
# funcao para identificar qual prompt usar
def identify_prompt(file_path):
    # pegar o diretório do arquivo
    prompt_directory = os.path.dirname(file_path)
    prompt_files = get_files_from_directory(prompt_directory, ".prompt")
    _system_message = "Você é um assistente de IA que classifica emissoras de rádio e suas respectivas praças."
    _prompt = f''' 
    Com base no nome do arquivo {file_path}, me diga qual o arquivo de prompt, 
    da lista a seguir é mais adequado para processar esse arquivo?
    {prompt_files}
    Retorne apenas o nome do arquivo, sem formatações ou explicações adicionais.
    Caso nenhum arquivo seja adequado, retorne "default.prompt".
    '''
    r = call_llm(GPT_4o, _system_message, _prompt, None, region=EASTUS)
    prompt_file = r["2.response"]
    return f"{prompt_directory}/{prompt_file}"

def processa_grade(file_path):
    # Extrair o texto da tabela do PDF
    table_as_text = extract_table_as_text_from_pdf(file_path)
    prompt_file = identify_prompt(file_path)
    prompt = read_prompt_from_file(prompt_file)

    _prompt = f''' 
        {prompt}
        {table_as_text}
    '''
    system_message = '''
    Você é um assistente de IA que ajuda a organizar informações em arquivos texto.
    Você deve analisar o texto fornecido e extrair as informações relevantes de forma organizada.
    '''
    r = call_llm(GPT_4o_mini, system_message, _prompt, None, region=EASTUS)
    return r["2.response"]

def salva_grade(arquivo, grade):
    nome_arquivo = os.path.splitext(os.path.basename(arquivo))[0] + ".html"
    diretorio = os.path.dirname(arquivo)
    caminho_arquivo = os.path.join(diretorio, nome_arquivo)
    save_file(grade, caminho_arquivo)
  
    
f = "C:/Users/rmendonca/OneDrive/_00/Matheus/TABELA BAND FM JANEIRO 2025.pdf"
salva_grade(f,processa_grade(f))

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


save_file C:/Users/rmendonca/OneDrive/_00/Matheus\TABELA BAND FM JANEIRO 2025.html


In [ ]:
diretorio = "C:/Users/rmendonca/OneDrive/_00/Matheus/"
arquivos = get_files_from_directory(diretorio, ".pdf")
for arquivo in arquivos:
    print(f"Processando arquivo: {arquivo}")
    resultado = processa_grade(arquivo)
    nome_arquivo = os.path.splitext(os.path.basename(arquivo))[0] + ".html"
    caminho_arquivo = os.path.join(diretorio, nome_arquivo)
    save_file(resultado, caminho_arquivo)